In [ ]:
# Download the remote submission to this computer.
import base64
from IPython.display import HTML, display

csv_bytes = Path('submission.csv').read_bytes()
assert csv_bytes, 'submission.csv is empty; rerun the generation cell first.'
csv_b64 = base64.b64encode(csv_bytes).decode('ascii')
display(HTML(
    '<a download="submission.csv" href="data:text/csv;base64,'
    + csv_b64
    + '">Download submission.csv to this computer</a>'
))

# Part 1 — T4 runner

Run this notebook in VS Code with a Colab T4 kernel. It uses the updated `p1_rag` pipeline and writes `submission.csv` in the repository root.

If the competition data is not under `data/`, set `DATA_DIR` in the configuration cell to the folder containing `Questions.csv`, `kb_manifest.json`, and the three PDFs.

In [22]:
%pip install -q -U kaggle

from pathlib import Path
from getpass import getpass
import os

token = getpass("Paste Kaggle token (leave blank if data is already available): ").strip()

if token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    token_file = kaggle_dir / "access_token"
    token_file.write_text(token)
    token_file.chmod(0o600)

    os.environ["KAGGLE_API_TOKEN"] = token
    print("Kaggle authentication configured")
else:
    print("Kaggle authentication skipped")

Kaggle authentication configured


In [23]:
from pathlib import Path
import os, subprocess, zipfile

assert os.environ.get("KAGGLE_API_TOKEN"), (
    "Kaggle token is not set. Run the authentication cell first."
)

download_dir = Path("/content/kaggle_download")
data_dir = Path("/content/competition_data")
download_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [
        "kaggle", "competitions", "download",
        "-c", "data-266-fall-2026-lab-2-part-1-multimodal-rag",
        "-p", str(download_dir),
    ],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Kaggle download failed; see stderr above.")

for archive in download_dir.glob("*.zip"):
    with zipfile.ZipFile(archive) as z:
        z.extractall(data_dir)

required = ["Questions.csv", "kb_manifest.json"]
missing = [name for name in required if not (data_dir / name).exists()]
assert not missing, f"Downloaded archive is missing: {missing}"
os.environ["DATA_DIR"] = str(data_dir)
print("Competition data ready at", data_dir)

Return code: 0



  0%|          | 0.00/1.38M [00:00<?, ?B/s]
100%|██████████| 1.38M/1.38M [00:00<00:00, 151MB/s]

Competition data ready at /content/competition_data


In [24]:
print("Kaggle download cell completed; continuing to repository setup.")

Kaggle download cell completed; continuing to repository setup.


In [25]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
# In a remote Colab kernel the local VS Code workspace is not visible. Clone the code there.
if not any((candidate_root / 'p1_rag').exists() and (candidate_root / 'requirements.txt').exists() for candidate_root in [ROOT, *ROOT.parents]):
    clone_root = Path('/content/DATA266_LAB2')
    if not (clone_root / 'p1_rag').exists():
        import subprocess
        subprocess.run(['git', 'clone', 'https://github.com/sivasuryachandran/DATA266_LAB2', str(clone_root)], check=True)
    ROOT = clone_root
# Find the repository even when the remote kernel starts several directories away.
for candidate_root in [ROOT, *ROOT.parents]:
    if (candidate_root / 'p1_rag').exists() and (candidate_root / 'requirements.txt').exists():
        ROOT = candidate_root
        break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Set this explicitly if your Colab data is mounted elsewhere.
DATA_DIR = os.environ.get('DATA_DIR', '')
if DATA_DIR:
    DATA_DIR = str(Path(DATA_DIR).expanduser().resolve())
else:
    candidates = [
        ROOT / 'data/.cache/competitions/data-266-fall-2026-lab-2-part-1-multimodal-rag',
        ROOT / 'data',
        Path('/content/data'),
        Path('/content/competition_data'),
        Path('/content/drive/MyDrive'),
    ]
    # Check direct candidates first, then search mounted folders for the two required files.
    DATA_DIR = next((str(p) for p in candidates if (p / 'Questions.csv').exists() and (p / 'kb_manifest.json').exists()), '')
    if not DATA_DIR:
        search_roots = [p for p in [ROOT, ROOT.parent, Path('/content'), Path('/content/drive/MyDrive')] if p.exists()]
        for base in search_roots:
            matches = base.rglob('Questions.csv')
            found = next((p.parent for p in matches if (p.parent / 'kb_manifest.json').exists()), None)
            if found:
                DATA_DIR = str(found)
                break

print('Repository:', ROOT)
print('Data directory:', DATA_DIR or 'not found')
if not DATA_DIR:
    print('Checked these locations:')
    for p in candidates:
        print(' -', p)
    raise FileNotFoundError('Competition data is not mounted. Set DATA_DIR to the folder containing Questions.csv and kb_manifest.json.')
assert Path(DATA_DIR, 'Questions.csv').exists(), f'Questions.csv is missing from {DATA_DIR}'
assert Path(DATA_DIR, 'kb_manifest.json').exists(), f'kb_manifest.json is missing from {DATA_DIR}'

Repository: /content/DATA266_LAB2
Data directory: /content/competition_data


In [26]:
# Run this once per environment. Restart the kernel after installing if VS Code requests it.
%pip install -q -r requirements.txt pandas
print('Dependencies installed')

Dependencies installed


In [27]:
# T4 configuration. The defaults are the models used by the project.
os.environ['DATA_DIR'] = DATA_DIR
os.environ.setdefault('P1_VLM', 'Qwen/Qwen2.5-VL-7B-Instruct')
os.environ.setdefault('P1_LLM', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit')
os.environ.setdefault('LLM_QUANT', '4bit')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print('P1_VLM =', os.environ['P1_VLM'])
print('P1_LLM =', os.environ['P1_LLM'])

P1_VLM = Qwen/Qwen2.5-VL-7B-Instruct
P1_LLM = unsloth/Qwen2.5-14B-Instruct-bnb-4bit


In [28]:
# Build the index and answer Questions.csv. Figure descriptions are cached in work_p1.
from p1_rag.pipeline import build, answer_file

questions_path = str(Path(DATA_DIR) / 'Questions.csv')
system, items, figures = build(data_dir=DATA_DIR, work_dir='work_p1')
submission = answer_file(system, questions_path, 'submission.csv', with_time=True)
out, timing = submission
print(f'Generated {len(out)} rows in {sum(timing.values())/60:.1f} minutes')
print('Saved:', Path('submission.csv').resolve())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

... (33 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarni

Generated 40 rows in 109.8 minutes
Saved: /content/DATA266_LAB2/submission.csv


In [29]:
# Validate the exact competition schema.
import subprocess
validator = Path(DATA_DIR) / 'validate_submission.py'
if validator.exists():
    result = subprocess.run([sys.executable, str(validator), 'submission.csv', questions_path], text=True, capture_output=True)
    print(result.stdout or result.stderr)
    result.check_returncode()
else:
    print('validate_submission.py not found; checking columns locally')
    expected = ['id', 'conversation_id', 'question_id', 'answer', 'figure_references', 'citations', 'status']
    assert list(out.columns) == expected
print('Submission schema is valid')

valid: 40 rows

Submission schema is valid


In [30]:
# Optional development score against the hand-written public key.
score_script = Path('tools/local_score.py')
if score_script.exists():
    result = subprocess.run([sys.executable, str(score_script), 'submission.csv'], text=True, capture_output=True)
    print(result.stdout)
else:
    print('Local scorer not present; use the competition leaderboard for the official score.')

Local scorer not present; use the competition leaderboard for the official score.


## Output

Download `submission.csv` from the repository root and upload it to the competition. Do not upload `work_p1/`; it is only the local/cache index.